In [1]:
import pandas as pd
df=pd.read_csv('/content/12. 100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
##words to numbers
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace('?','')    #data preprocessing
  text=text.replace("'","")

  return text.split()

In [3]:
#vocab
vocab={'<UNK>': 0}   #for unknown ques

def build_vocab(row):

  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  merged_tokens=tokenized_question+tokenized_answer
  print(merged_tokens)

  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)



In [4]:
df.apply(build_vocab,axis=1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [5]:
len(vocab)

324

In [6]:
#conv words to num indices
def text_to_indices(text,vocab):
  indexed_text=[]

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token]) #adding index of word from vocab
    else:
      indexed_text.append(vocab['<UNK>'])  #ret 0 for unindexed words
  return indexed_text



In [7]:
text_to_indices("what is campusx",vocab)

[1, 2, 0]

In [8]:
import torch
from torch.utils.data import Dataset,DataLoader


In [9]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)


In [10]:
dataset=QADataset(df,vocab)

In [11]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [12]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [23]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[ 42, 101,   2,   3,  17]]) tensor([102])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([36])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([36])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([95])
tensor([[ 10,  11, 189, 158, 190]]) tensor([191])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[  1,   2,   3,   4,   5, 236, 237]]) tensor([238])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([41])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([72])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([61])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([307])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([121])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([162])
ten

In [14]:
import torch.nn as nn

In [19]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embeddding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question= self.embeddding(question)
    hidden,final= self.rnn(embedded_question)
    output= self.fc(final.squeeze(0))

    return output

In [28]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
#to prevent swap of batchlen and no of words in sentence swap

z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))  #removes batchlen dim

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [25]:
#define lr & epoch
learning_rate=0.001
epochs=50

In [26]:
#define model,loss & optimizer
model=SimpleRNN(len(vocab))

criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [27]:
#training loop
for epoch in range(epochs):
  total_loss=0

  for question,answer in dataloader:
    optimizer.zero_grad()

    #model
    output=model(question)
    #loss-> output shape(1,324),ans-(1)
    loss=criterion(output,answer[0])
    #backward pass
    loss.backward()
    #gradient update
    optimizer.step()

    total_loss+=loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 526.820846
Epoch: 2, Loss: 458.116175
Epoch: 3, Loss: 380.413778
Epoch: 4, Loss: 318.864490
Epoch: 5, Loss: 267.958624
Epoch: 6, Loss: 220.555818
Epoch: 7, Loss: 177.386461
Epoch: 8, Loss: 139.024708
Epoch: 9, Loss: 107.137242
Epoch: 10, Loss: 83.070343
Epoch: 11, Loss: 63.805831
Epoch: 12, Loss: 50.584741
Epoch: 13, Loss: 40.347924
Epoch: 14, Loss: 32.489752
Epoch: 15, Loss: 26.807446
Epoch: 16, Loss: 22.283170
Epoch: 17, Loss: 18.754802
Epoch: 18, Loss: 16.218974
Epoch: 19, Loss: 13.623800
Epoch: 20, Loss: 11.729155
Epoch: 21, Loss: 10.260961
Epoch: 22, Loss: 9.037165
Epoch: 23, Loss: 8.039189
Epoch: 24, Loss: 7.202258
Epoch: 25, Loss: 6.461402
Epoch: 26, Loss: 5.800967
Epoch: 27, Loss: 5.281002
Epoch: 28, Loss: 4.771619
Epoch: 29, Loss: 4.370142
Epoch: 30, Loss: 3.999960
Epoch: 31, Loss: 3.674305
Epoch: 32, Loss: 3.366413
Epoch: 33, Loss: 3.114145
Epoch: 34, Loss: 2.880147
Epoch: 35, Loss: 2.671010
Epoch: 36, Loss: 2.477272
Epoch: 37, Loss: 2.306431
Epoch: 38, Loss: 

In [47]:
def predict(model,question,threshold=0.5):

  #conv que to no.
  numerical_question= text_to_indices(question,vocab)
  #tensor
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)
  #send to model
  output=model(question_tensor)

  #conv logits to probab
  probability=torch.nn.functional.softmax(output,dim=1)

  #find index of max probab
  value,index=torch.max(probability,dim=1)


  if value< threshold:
    print("I dont know")

  print(list(vocab.keys())[index])



In [48]:
predict(model,"capital of france")

paris


In [43]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit